# IAA Hackathon 2026 — Notebook 3: GLM & Blend

**Purpose:** Fit Tweedie GLM using top features from XGBoost, predict on 2025 test files, blend with XGBoost, generate final submissions.

**AI-assisted:** Genie Code generated GLM code, one-hot encoding, blend logic, bias correction.
**Human decisions:** Feature selection guided by XGBoost importance, blend weights (50/50 and 70/30), trend uplift approach.

In [0]:
%pip install -q statsmodels openpyxl

import pandas as pd
import numpy as np
import os
import statsmodels.api as sm
import gc

DATA_DIR = '../data'
PROC_DIR = '../data/proc'
EXCEL_DIR = '../Excel files_v3'
SUB_DIR = '../submissions_v3'

# Read XGBoost feature importance FIRST to determine needed columns
TOP_N = 30
feature_importance = {}
needed_cols_train = {}
needed_cols_test = {}
for line in ['motor', 'buildings', 'contents']:
    feature_importance[line] = pd.read_excel(f'{EXCEL_DIR}/xgboost_model_checkpoint.xlsx', sheet_name=f'{line}_top30')
    top_feats = feature_importance[line].head(TOP_N)['feature'].tolist()
    if 'Year' not in top_feats:
        top_feats.append('Year')
    needed_cols_train[line] = list(dict.fromkeys(top_feats + ['ultimate_claims', 'exposure', 'row_id']))
    needed_cols_test[line] = list(dict.fromkeys(top_feats + ['exposure', 'row_id']))

# Load only needed columns (memory-efficient — ~5x less memory)
train = {}
test = {}
for line in ['motor', 'buildings', 'contents']:
    train[line] = pd.read_parquet(f'{PROC_DIR}/train_{line}.parquet', columns=needed_cols_train[line])
    test[line] = pd.read_parquet(f'{PROC_DIR}/test_{line}.parquet', columns=needed_cols_test[line])
    print(f'{line}: train {train[line].shape} | test {test[line].shape}')

sample_submission = pd.read_parquet(f'{PROC_DIR}/sample_submission.parquet')
xgb_sub = pd.read_csv(f'{SUB_DIR}/xgboost_tweedie_v1.csv')
print(f'XGBoost submission: {len(xgb_sub):,} rows')
for line in ['motor', 'buildings', 'contents']:
    print(f'  {line} top feature: {feature_importance[line].iloc[0]["feature"]}')

In [0]:
# --- Fit Tweedie GLM and Predict ---
# Uses XGBoost feature importance, filtered by cardinality
# Keeps numerics + low-cardinality categoricals (<= 10 levels), caps at 12 per line
# High-cardinality categoricals target-encoded with Bayesian shrinkage
# statsmodels GLM with proper offset (log(exposure)) — standard actuarial approach

from sklearn.preprocessing import StandardScaler

def tweedie_deviance(y_true, y_pred, p=1.5):
    y = np.asarray(y_true, dtype=float)
    mu = np.maximum(np.asarray(y_pred, dtype=float), 1e-10)
    term = np.where(y > 0,
                   y**(2-p) / ((1-p)*(2-p)) - y * mu**(1-p) / (1-p) + mu**(2-p) / (2-p),
                   mu**(2-p) / (2-p))
    return 2 * np.sum(term)

def tweedie_d2(y_true, y_pred, p=1.5):
    dev_model = tweedie_deviance(y_true, y_pred, p)
    dev_null = tweedie_deviance(y_true, np.mean(y_true), p)
    return 1 - dev_model / dev_null if dev_null > 0 else 0

# Feature selection: XGBoost importance filtered by cardinality
# Keep numerics + low-cardinality categoricals (<= 10 levels), cap at 15 features
# This prevents one-hot blowup (e.g. make_model created 300+ columns for motor)
MAX_FEATS = 15
MAX_CARDINALITY = 10

glm_test_preds = {}
glm_val_scores = {}
glm_n_features = {}
test_row_ids = {}
glm_selected_features = {}

# Toggle: apply CAGR trend uplift for 2025 projection
TREND_UPLIFT = False

for line in ['motor', 'buildings', 'contents']:
    # Get top 30 from XGBoost importance
    all_top = feature_importance[line].head(30)['feature'].tolist()
    if 'Year' not in all_top:
        all_top.append('Year')
    
    # Load all top 20 columns + targets
    cols_tr = list(dict.fromkeys(all_top + ['ultimate_claims', 'exposure', 'row_id']))
    cols_te = list(dict.fromkeys(all_top + ['exposure', 'row_id']))
    tr_full = train[line][cols_tr].copy()
    te_full = test[line][cols_te].copy()
    del train[line], test[line]
    gc.collect()
    
    # Target-encode high-cardinality categoricals (e.g. make_model, body_type)
    # Converts to numeric (smoothed mean ultimate_claims per category)
    # Computed on training split only (2021-2023) to avoid leakage from validation
    train_split_mask = tr_full['Year'] <= 2023
    overall_mean = tr_full.loc[train_split_mask, 'ultimate_claims'].mean()
    SMOOTHING = 30  # Bayesian shrinkage — pull rare categories toward overall mean
    
    te_applied = []
    for f in all_top[:]:
        if f == 'Year':
            continue
        if tr_full[f].dtype in ('object', 'bool') and tr_full[f].nunique() > MAX_CARDINALITY:
            te_map = tr_full.loc[train_split_mask].groupby(f)['ultimate_claims'].mean()
            cat_counts = tr_full.loc[train_split_mask].groupby(f).size()
            smooth_mean = (te_map * cat_counts + overall_mean * SMOOTHING) / (cat_counts + SMOOTHING)
            te_col = f + '_te'
            tr_full[te_col] = tr_full[f].map(smooth_mean).fillna(overall_mean).astype(np.float32)
            te_full[te_col] = te_full[f].map(smooth_mean).fillna(overall_mean).astype(np.float32)
            all_top = [te_col if x == f else x for x in all_top]
            te_applied.append(f'{f} -> {te_col}')
    
    if te_applied:
        print(f'  {line}: target-encoded {te_applied}')
    
    # Filter: keep numerics + low-cardinality categoricals, cap at MAX_FEATS
    selected = []
    for f in all_top:
        if f == 'Year':
            selected.append(f)
            continue
        if tr_full[f].dtype in ('object', 'bool'):
            if tr_full[f].nunique() <= MAX_CARDINALITY:
                selected.append(f)
        else:
            selected.append(f)
        if len(selected) >= MAX_FEATS:
            break
    if 'Year' not in selected:
        selected.insert(0, 'Year')
    
    top_feats = selected
    glm_selected_features[line] = top_feats
    
    # Keep only selected features
    keep_tr = list(dict.fromkeys(top_feats + ['ultimate_claims', 'exposure', 'row_id']))
    keep_te = list(dict.fromkeys(top_feats + ['exposure', 'row_id']))
    tr = tr_full[keep_tr].copy()
    te = te_full[keep_te].copy()
    del tr_full, te_full
    gc.collect()
    
    # Identify categorical vs numerical
    cat_feats = [f for f in top_feats if tr[f].dtype == 'object' or tr[f].dtype == 'bool']
    num_feats = [f for f in top_feats if f not in cat_feats]
    
    # Impute missing values
    for col in num_feats:
        med = tr[col].median()
        tr[col] = tr[col].fillna(med)
        te[col] = te[col].fillna(med)
    for col in cat_feats:
        mode_val = tr[col].astype(str).mode()[0] if not tr[col].astype(str).mode().empty else 'missing'
        tr[col] = tr[col].astype(str).fillna(mode_val)
        te[col] = te[col].astype(str).fillna(mode_val)
    
    # One-hot encode categoricals
    tr_enc = pd.get_dummies(tr, columns=cat_feats, drop_first=True, dtype=np.float32)
    te_enc = pd.get_dummies(te, columns=cat_feats, drop_first=True, dtype=np.float32)
    te_enc = te_enc.reindex(columns=tr_enc.columns, fill_value=0)
    
    # Offset: log(exposure) — proper actuarial offset (not a regularized feature)
    off_tr = np.log(tr_enc['exposure'].clip(lower=1e-6)).values
    off_te = np.log(te_enc['exposure'].clip(lower=1e-6)).values
    
    # Scale numeric features for convergence (exclude Year — needed for splitting)
    num_cols = [c for c in num_feats if c in tr_enc.columns and c != 'Year']
    if num_cols:
        scaler = StandardScaler()
        tr_enc[num_cols] = scaler.fit_transform(tr_enc[num_cols]).astype(np.float32)
        te_enc[num_cols] = scaler.transform(te_enc[num_cols]).astype(np.float32)
    
    # Feature columns (exclude targets and metadata)
    non_feat = ['ultimate_claims', 'exposure', 'row_id']
    feat_cols = [c for c in tr_enc.columns if c not in non_feat]
    glm_n_features[line] = len(feat_cols)
    
    # Add intercept for statsmodels
    tr_enc['const'] = 1.0
    te_enc['const'] = 1.0
    feat_cols = feat_cols + ['const']
    
    # Time-based split: train 2021-2023, validate 2024
    train_mask = tr_enc['Year'] <= 2023
    val_mask = tr_enc['Year'] == 2024
    
    X_train = tr_enc.loc[train_mask, feat_cols].values.astype(np.float64)
    y_train = tr_enc.loc[train_mask, 'ultimate_claims'].values.astype(np.float64)
    off_train = off_tr[train_mask.values]
    X_val = tr_enc.loc[val_mask, feat_cols].values.astype(np.float64)
    y_val = tr_enc.loc[val_mask, 'ultimate_claims'].values.astype(np.float64)
    off_val = off_tr[val_mask.values]
    
    # Fit statsmodels Tweedie GLM with proper offset (IRLS — more stable than lbfgs)
    glm = sm.GLM(
        y_train, X_train,
        family=sm.families.Tweedie(link=sm.families.links.Log(), var_power=1.5),
        offset=off_train
    )
    result = glm.fit(max_iter=100)
    
    # Validation
    val_pred = np.maximum(result.predict(X_val, offset=off_val), 0)
    score = tweedie_d2(y_val, val_pred, p=1.5)
    glm_val_scores[line] = score
    
    # Bias correction from validation
    bias_factor = y_val.mean() / max(val_pred.mean(), 1e-6)
    
    # Trend uplift (CAGR from 2021 to 2024, applied for 2025)
    yr_stats = tr_enc.groupby('Year').agg(ult=('ultimate_claims', 'sum'), exp=('exposure', 'sum'))
    rate_first = yr_stats['ult'].iloc[0] / yr_stats['exp'].iloc[0]
    rate_last = yr_stats['ult'].iloc[-1] / yr_stats['exp'].iloc[-1]
    cagr = (rate_last / rate_first) ** (1/(len(yr_stats)-1)) - 1 if len(yr_stats) >= 2 else 0
    trend_factor = 1 + cagr if TREND_UPLIFT else 1.0
    
    # Test prediction
    X_test = te_enc[feat_cols].values.astype(np.float64)
    pred = np.maximum(result.predict(X_test, offset=off_te), 0)
    pred = pred * bias_factor * trend_factor
    pred = np.maximum(pred, 0.01)
    glm_test_preds[line] = pred
    
    # Save row_ids for blend cell, then clean up memory
    test_row_ids[line] = te['row_id'].values
    del tr, te, tr_enc, te_enc, X_train, X_val, y_train, y_val, off_train, off_val, glm, result, X_test
    gc.collect()
    
    print(f'{line}: GLM D² = {score:.4f} | bias={bias_factor:.3f} | CAGR={cagr*100:.1f}% | '
          f'mean: {pred.mean():.2f} | n_cols: {len(feat_cols)} | features: {top_feats}')

print(f'\nAverage GLM D²: {np.mean(list(glm_val_scores.values())):.4f}')

In [0]:
# --- Blend XGBoost + GLM and Generate Submissions ---

# Build GLM submission
glm_preds_df = pd.concat([
    pd.DataFrame({'row_id': test_row_ids[line], 'prediction': glm_test_preds[line]})
    for line in ['motor', 'buildings', 'contents']
], ignore_index=True)

glm_sub = sample_submission[['row_id']].merge(glm_preds_df, on='row_id', how='left')
assert glm_sub['prediction'].notna().all() and (glm_sub['prediction'] >= 0.01).all()

glm_sub.to_csv(f'{SUB_DIR}/glm_tweedie_v1.csv', index=False, float_format='%.2f')
print(f'GLM submission saved: {SUB_DIR}/glm_tweedie_v1.csv')

# Blend: merge XGBoost and GLM predictions
blend_df = xgb_sub[['row_id']].copy()
blend_df = blend_df.merge(xgb_sub[['row_id', 'prediction']].rename(columns={'prediction': 'xgb'}), on='row_id')
blend_df = blend_df.merge(glm_sub[['row_id', 'prediction']].rename(columns={'prediction': 'glm'}), on='row_id')

# 50/50 blend
blend_50 = blend_df[['row_id']].copy()
blend_50['prediction'] = (0.5 * blend_df['xgb'] + 0.5 * blend_df['glm']).clip(lower=0.01)
blend_50.to_csv(f'{SUB_DIR}/blend_5050_v1.csv', index=False, float_format='%.2f')

# 70/30 blend (XGBoost-weighted)
blend_70 = blend_df[['row_id']].copy()
blend_70['prediction'] = (0.7 * blend_df['xgb'] + 0.3 * blend_df['glm']).clip(lower=0.01)
blend_70.to_csv(f'{SUB_DIR}/blend_7030_v1.csv', index=False, float_format='%.2f')

print(f'Blend 50/50 saved (mean: {blend_50["prediction"].mean():.2f})')
print(f'Blend 70/30 saved (mean: {blend_70["prediction"].mean():.2f})')

# Summary by line
print(f'\n{"Line":<12} {"XGB":>8} {"GLM":>8} {"Blend50":>8} {"Blend70":>8}')
for prefix, name in [('M', 'motor'), ('B', 'buildings'), ('C', 'contents')]:
    x = xgb_sub.loc[xgb_sub['row_id'].str[0] == prefix, 'prediction'].mean()
    g = glm_sub.loc[glm_sub['row_id'].str[0] == prefix, 'prediction'].mean()
    b5 = blend_50.loc[blend_50['row_id'].str[0] == prefix, 'prediction'].mean()
    b7 = blend_70.loc[blend_70['row_id'].str[0] == prefix, 'prediction'].mean()
    print(f'{name:<12} {x:>8.1f} {g:>8.1f} {b5:>8.1f} {b7:>8.1f}')

In [0]:
# --- Excel Checkpoint: Model Comparison ---

# Read XGBoost scores from Excel
xgb_metrics = pd.read_excel(f'{EXCEL_DIR}/xgboost_model_checkpoint.xlsx', sheet_name='Model Metrics')

comparison_rows = []
for line in ['motor', 'buildings', 'contents']:
    xgb_score = xgb_metrics.loc[xgb_metrics['line'] == line, 'tweedie_d2'].values[0]
    comparison_rows.append({
        'line': line,
        'xgb_d2': round(xgb_score, 4),
        'glm_d2': round(glm_val_scores[line], 4),
        'glm_n_features': glm_n_features[line],
        'glm_features': ', '.join(glm_selected_features[line]),
        'xgb_mean_pred': round(xgb_sub.loc[xgb_sub['row_id'].str[0] == {'motor': 'M', 'buildings': 'B', 'contents': 'C'}[line], 'prediction'].mean(), 2),
        'glm_mean_pred': round(glm_test_preds[line].mean(), 2),
    })
comparison_df = pd.DataFrame(comparison_rows)

blend_summary = pd.DataFrame({
    'model': ['XGBoost', 'GLM', 'Blend 50/50', 'Blend 70/30'],
    'mean_pred': [xgb_sub['prediction'].mean(), glm_sub['prediction'].mean(),
                  blend_50['prediction'].mean(), blend_70['prediction'].mean()]
})

excel_path = f'{EXCEL_DIR}/glm_blend_checkpoint.xlsx'
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    comparison_df.to_excel(writer, sheet_name='Model Comparison', index=False)
    blend_summary.to_excel(writer, sheet_name='Blend Summary', index=False)

print(f'Excel checkpoint saved: {excel_path}')
print(f'\nModel Comparison:')
print(comparison_df.to_string(index=False))
print(f'\nBlend Summary:')
print(blend_summary.to_string(index=False))

## Summary

### GLM
- Feature selection: data-driven from XGBoost importance (top 30, filtered by cardinality)
- High-cardinality categoricals (e.g. make_model) target-encoded with Bayesian shrinkage (SMOOTHING=30)
- Target encoding computed on training split only (2021-2023) to avoid leakage
- statsmodels GLM with IRLS (more stable than sklearn lbfgs), proper offset for exposure
- One-hot encoded low-cardinality categoricals, log(exposure) as offset
- Same bias correction and trend uplift as XGBoost
- GLM may extrapolate Year trend better than XGBoost (linear on log scale vs tree splits)

### Blend
- 50/50 blend: simple average of XGBoost and GLM
- 70/30 blend: XGBoost-weighted (captures non-linear effects better)

### Submissions (in submissions_v3/)
- `xgboost_tweedie_v1.csv` — XGBoost only
- `glm_tweedie_v1.csv` — GLM only
- `blend_5050_v1.csv` — 50/50 blend (recommended first submission)
- `blend_7030_v1.csv` — 70/30 XGBoost-weighted blend

### Future Improvements (noted for later)
- Hypertune XGBoost and GLM parameters
- K-fold cross-validation
- Separate frequency and severity GLMs
- Group ages/years licensed to prevent overfitting
- Correlation analysis, hypothesis testing, p-values for rating factors